<a href="https://colab.research.google.com/github/ranmaru1115/MorningBot/blob/main/MorningBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌤️ モーニングボット - LINE天気通知 (Colab版)
設定を変更して、左側の再生ボタン（三角マーク）を押すだけで実行できます。

In [ ]:
# ==========================================
# 🛠️ 設定エリア（ここを書き換えてください）
# ==========================================

# LINE Developersの「Messaging API設定」にある長いトークンを入れてください
ACCESS_TOKEN = "YOUR_LINE_CHANNEL_ACCESS_TOKEN"

# 送信先のLINEユーザーIDまたはグループIDを入れてください（Cから始まる文字列など）
GROUP_ID = "YOUR_LINE_GROUP_ID"

# 天気を取得したい地域のコード（デフォルトは東京：130010）
AREA_CODE = "130010"

In [ ]:
# ==========================================
# ⚙️ 実行処理（ここは触らなくてOKです）
# ==========================================
import json
import requests


def get_weather_data(city_code):
    weather_url = (
        f"https://weather.tsukumijima.net/api/forecast/city/{city_code}"
    )
    try:
        response = requests.get(weather_url)
        response.raise_for_status()
        weather_data = response.json()
        title = weather_data.get("title", "お天気予報")
        forecasts = weather_data.get("forecasts", [])
        if not forecasts:
            return "天気データが空です。"

        today = forecasts[0]
        temp_max_data = today.get("temperature", {}).get("max", {})
        temp_max = (
            temp_max_data.get("celsius")
            if temp_max_data and temp_max_data.get("celsius")
            else "--"
        )

        msg = f"【{title}】\n{today.get('dateLabel', '今日')}の天気: {today.get('telop', '不明')}\n最高気温: {temp_max}度\n\n今日も一日頑張りましょう！\n※これだけを頼りにしないでね"
        return msg
    except Exception as e:
        return f"天気情報取得エラー: {e}"


def send_line_message(text_content):
    line_url = "https://api.line.me/v2/bot/message/push"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {ACCESS_TOKEN}",
    }
    payload = {
        "to": GROUP_ID,
        "messages": [{"type": "text", "text": text_content}],
    }
    try:
        response = requests.post(
            line_url, headers=headers, data=json.dumps(payload)
        )
        if response.status_code == 200:
            print("🎉 LINEへの天気送信に成功しました！")
        else:
            print(f"❌ LINEエラー（コード: {response.status_code}）\n{response.text}")
    except Exception as e:
        print(f"通信エラー: {e}")


# 実行
print("🌤️ 天気予報を取得中...")
weather_message = get_weather_data(AREA_CODE)
send_line_message(weather_message)